In [21]:
from dotenv import load_dotenv
load_dotenv()


True

In [22]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_core.prompts import PromptTemplate

 

In [23]:
loader=PyPDFLoader("../data/PdfData.pdf")
pdfData=loader.load()

splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
splittedData=splitter.split_documents(pdfData)

embeddings=OpenAIEmbeddings(model="text-embedding-3-large")

vector_store=Chroma.from_documents(
    documents=splittedData,
    embedding=embeddings,
)


In [25]:
llm=ChatOpenAI(model="gpt-5")
def get_context(query:str):
   res=vector_store.similarity_search(query=query,k=2)
   context=""
   for doc in res:
     context+=doc.page_content+"\n"

   return {
       "context":context,
       "question":query
   }

In [26]:
prompt=PromptTemplate.from_template(
    """
        You are a helpful assistent and provides answers based on the context for user 
        question and if you do not know the answer then you can say that i do not know.
        
        Context:{context}
        Question:{question}
    """
)


In [33]:
Rag_Chain= get_context | prompt | llm 
response=Rag_Chain.invoke("who is the pm of india ")


In [34]:
print(response)

content='I don’t know based on the provided context.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 339, 'prompt_tokens': 130, 'total_tokens': 469, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 320, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-ELshqrFsO0UU3NE4cI2OgdkAYY5fC', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--01a081c9-dabf-7c40-955a-8742e902ad6c-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 130, 'output_tokens': 339, 'total_tokens': 469, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 320}}
